In [ ]:
# v17 setup. Run before the smoke cell.
#
# Generation runs on the API, so it does not contend with anything on the GPU.
#
# The check that matters is per-frame adherence, not F1. v17 introduces seven negative
# constructions no previous generator produced (denial, study-only, incompatibility,
# enzyme-only, out-of-scope, sequential, title) and three positive ones (mechanism_mixed,
# effect_pd, appositive). Whether the model can actually write them is measurable from
# 300 sentences and does not need a training run.

import os
import json
import importlib
import re
import random
from collections import Counter, defaultdict

from openai import OpenAI

import ddi.prompt, ddi.resolve, ddi.gates, ddi.synth, ddi.divergence
for m in (ddi.synth, ddi.prompt, ddi.resolve, ddi.gates, ddi.divergence):
    importlib.reload(m)

from ddi.data import build_human
from ddi.vocab import build_vocab
from ddi.prompt import make_specs, render, make_sample_fn, fingerprint, FRAMES
from ddi.resolve import v14_sample_to_instances, generation_records
from ddi.synth import generate_raw, build_dataset_from_raw, RAW
from ddi.manifest import load_dataset
from ddi import gates, divergence as dv

MODEL, API, EFFORT = "gpt-oss-120b", "responses", "low"
V14_ID = "20260807-123340-ff79db"

client = OpenAI(base_url="http://api.llm.apps.os.dcs.gla.ac.uk/v1",
                api_key=os.environ["IDA_LLM_API_KEY"], max_retries=5, timeout=60.0)

vocab = build_vocab()
train, dev, val = build_human()
# the eight enumeration sentences carry 43% of instances and almost no positives, so
# they distort every aggregate: unfiltered positive rate 0.096, filtered 0.162, and they
# cost only 0.010 F1 to remove (0.800 -> 0.790)
per_sent = Counter(r["sent_id"] for r in train)
train_f = [r for r in train if per_sent[r["sent_id"]] < 190]
v14, _ = load_dataset(V14_ID)

print(f"prompt sha {fingerprint()}")
print(f"{len(FRAMES)} frames, weights sum {sum(f['w'] for f in FRAMES.values()):.3f}")
print(f"vocab {len(vocab.drugs)} drugs, {len(vocab.groups)} groups")
print(f"human filtered {len(train_f)} instances, v14 {len(v14)}")

# v17 renamed the entry points. If the aliases are still bound to an older module the
# smoke run will silently generate the wrong thing.
assert "frame" in make_specs(1, vocab=vocab, seed=0)[0], "make_specs is not v17"
assert "says" in make_specs(1, vocab=vocab, seed=0)[0], "make_specs is not v17"


# ---------------------------------------------------------------------------
# Render a few before spending anything. Free.
#
# What to look for: `says` must be rows of nouns, never a finished clause. The first
# v17 draft rendered "that X changes urinary elimination of Y, making steady-state
# levels lower", which the model strips "that" from and returns verbatim. That is the
# v13 content brief, removed in v14 for exactly this reason.
# ---------------------------------------------------------------------------
specs = make_specs(40, vocab=vocab, seed=0)
print("\n" + str(Counter(s["frame"] for s in specs).most_common()))

for want in ["regimen", "appositive", "mechanism_mixed", "denial", "title"]:
    s = next((x for x in specs if x["frame"] == want), None)
    if s:
        print(f"\n{'-' * 60}\n[{want}]")
        print(render(s))

In [ ]:

# ===========================================================================
# Cell 1: render zero-assertion specs and read them. Free.
# Every one of these previously licensed a joint-outcome claim.
# ===========================================================================
specs = make_v14_specs(40, vocab=vocab, seed=0, composition="prior")
for s in specs:
    print("\n" + "-" * 60)
    print(render_v14(s))


In [ ]:
# v17 smoke: 300 specs, then read the output frame by frame.
#
# v17 has 20 frames, 7 of them negative constructions that no previous generator
# produced. The old joint-outcome regex is kept but it is now only one of several
# checks, and it applies to the `regimen` frame specifically rather than to all
# zero-assertion specs, because v17's other negative frames DO assert things (a denial,
# a study, an incompatibility) while still being gold NONE.
#
# Read the output. The per-frame breakdown says which constructions the model can
# actually write; nothing else in the pipeline measures that.

import json
import re
from collections import Counter, defaultdict

from ddi.prompt import make_specs, render, make_sample_fn, fingerprint, FRAMES
from ddi.resolve import v14_sample_to_instances, generation_records
from ddi.synth import generate_raw, build_dataset_from_raw, RAW
from ddi.manifest import load_dataset
from ddi import gates

GEN = "v17-check"
MODEL, API, EFFORT = "gpt-oss-120b", "responses", "low"
V14_ID = "20260807-123340-ff79db"

specs = make_specs(300, vocab=vocab, seed=0)
print(f"prompt sha {fingerprint()}")
print(Counter(s["frame"] for s in specs).most_common())

generate_raw(specs, make_sample_fn(client, model=MODEL, reasoning_effort=EFFORT,
                                   api=API),
             gen_id=GEN, max_workers=16)

did, stats = build_dataset_from_raw(
    GEN, resolver=v14_sample_to_instances, mode="markers",
    generator={"prompt_sha": fingerprint(), "model": MODEL, "version": "v17",
               "note": "frames: negatives as constructions, slot-based says, "
                       "frame-filtered axes"},
    vocab_source=vocab.fingerprint(), seed=0)
print(stats["reject_reasons"])
inst, _ = load_dataset(did)
v14, _ = load_dataset(V14_ID)



In [ ]:

# ---------------------------------------------------------------------------
# 1. Load the raw output keyed by frame. generate_raw stores the spec alongside the
#    sample, so frame membership survives.
# ---------------------------------------------------------------------------
MARK = re.compile(r"\[/?E[12]\]")
out = defaultdict(list)
n_err = 0
for line in (RAW / f"{GEN}.jsonl").read_text().splitlines():
    if not line:
        continue
    r = json.loads(line)
    if r.get("error") or not r.get("sample"):
        n_err += 1
        continue
    out[r["spec"]["frame"]].append((r["spec"], r["sample"]["sentence"]))
print(f"\n{n_err} errored, {sum(len(v) for v in out.values())} sentences")



In [ ]:

# ---------------------------------------------------------------------------
# 2. Frame adherence. Each negative frame must contain its marker and must NOT contain
#    the thing that would make it a positive. These are crude regexes: a smoke alarm,
#    not a measurement. Read the failures rather than trusting the rate.
# ---------------------------------------------------------------------------
PK_WORDS = re.compile(
    r"\b(concentration|level|AUC|Cmax|half-?life|clearance|absorb\w+|absorption|"
    r"metaboli\w+|bioavailab\w+|exposure|protein binding)\b", re.I)
ASSERTS = re.compile(
    r"\b(increas\w+|decreas\w+|reduc\w+|rais\w+|lower\w+|inhibit\w+|induc\w+|"
    r"potentiat\w+|antagonis\w+|interact\w+|result\w+ in|led to|caus\w+)\b", re.I)

MUST = {
    "denial":       re.compile(r"\b(no|not|without|unchanged|unaffected|neither)\b", re.I),
    "study_only":   re.compile(r"\b(stud\w+|investigat\w+|examin\w+|evaluat\w+|assess\w+)\b", re.I),
    "incompatible": re.compile(r"\b(mix\w+|precipitat\w+|solution|infusion|syringe|"
                               r"cloud\w+|unstable|instability)\b", re.I),
    "enzyme_only":  re.compile(r"\b(CYP\w*|P-?glycoprotein|UGT\w*|OATP\w*)\b", re.I),
    "out_of_scope": re.compile(r"\b(juice|wort|meal|tea|milk|smoking|liquorice|ginkgo)\b", re.I),
    "sequential":   re.compile(r"\b(after|before|stopp\w+|discontinu\w+|withdraw\w+|"
                               r"washout|subsequent\w*|previously)\b", re.I),
    "effect_pd":    re.compile(r"\b(synerg\w+|additive|potentiat\w+|antagoni\w+|"
                               r"enhanc\w+|oppos\w+|blunt\w+)\b", re.I),
    "appositive":   re.compile(r"\b(including|such as)\b", re.I),
    "title":        re.compile(r"^[^.]{0,60}:", re.I),
    "contradictory": re.compile(r"\b(although|however|not been reported|nevertheless|"
                                r"even though|despite)\b", re.I),
    "effect_protect": re.compile(r"\b(protect\w+|reduc\w+|less\b|lower\b|attenuat\w+)\b", re.I),
}
MUST_NOT = {
    # gold NONE frames that must not read as an interaction
    "study_only":   ASSERTS,
    "regimen":      ASSERTS,
    "incompatible": re.compile(r"\b(plasma|serum|absorption|metaboli\w+|clearance|"
                               r"in the body|systemic)\b", re.I),
    # priority order §4.5.11: an EFFECT sentence mentioning PK becomes MECHANISM
    "effect":         PK_WORDS,
    "effect_pd":      PK_WORDS,
    "effect_protect": PK_WORDS,
    "effect_failure": PK_WORDS,
    "advise":         PK_WORDS,
    "advise_reason":  PK_WORDS,
    "coordinate":     PK_WORDS,
    "appositive":     PK_WORDS,
}

print(f"\n{'frame':<18} {'n':>4} {'has marker':>11} {'breaks avoid':>13}")
fails = defaultdict(list)
for frame in sorted(out):
    rows = out[frame]
    must, mustnot = MUST.get(frame), MUST_NOT.get(frame)
    hit = miss = 0
    for spec, txt in rows:
        t = txt
        ok_m = bool(must.search(t)) if must else None
        ok_n = bool(mustnot.search(t)) if mustnot else None
        if ok_m is False:
            fails[frame].append(("no marker", t))
        if ok_n is True:
            fails[frame].append(("breaks avoid", t))
        hit += bool(ok_m)
        miss += bool(ok_n)
    m = f"{hit / len(rows):.2f}" if must else "-"
    n = f"{miss / len(rows):.2f}" if mustnot else "-"
    print(f"{frame:<18} {len(rows):>4} {m:>11} {n:>13}")


In [ ]:


# ---------------------------------------------------------------------------
# 3. The joint-outcome check, now scoped to `regimen`.
#    This is the failure pruning removed: 1,056 v14 NONE pairs claiming something about
#    the combination without claiming anything about a pair. Removing exactly those
#    gained 0.041 F1 over removing the same number at random.
#    v17's other negative frames legitimately assert things, so applying this to all of
#    them would flag correct output.
# ---------------------------------------------------------------------------
JOINT = re.compile(
    r"\b(combination|regimen|combined|together)\b[^.]{0,80}"
    r"\b(tolerat\w+|benefit\w*|improv\w+|effective|efficac\w+|additive|safe\w*|"
    r"outcome\w*|response)\b"
    r"|\b(suggest\w+|observ\w+|appear\w+|indicat\w+)\b[^.]{0,60}"
    r"\b(benefit\w*|improv\w+|efficac\w+|additive)\b", re.I)


def joint_rate(instances, name):
    """v14 baseline: zero-assertion sentences only, identified by having no positive
    pair anywhere in the sentence."""
    seen, hits = set(), 0
    has_pos = {r["sent_id"] for r in instances if r["label"] != "NONE"}
    for r in instances:
        if r["sent_id"] in seen or r["sent_id"] in has_pos:
            continue
        seen.add(r["sent_id"])
        hits += bool(JOINT.search(MARK.sub("", r["text"])))
    print(f"{name:<16} {hits}/{len(seen)} = {hits / max(len(seen), 1):.3f}")
    return hits / max(len(seen), 1)


print("\njoint-outcome claims in sentences that assert nothing")
joint_rate(v14, "v14 (all)")
reg = out.get("regimen", [])
h = sum(bool(JOINT.search(t)) for _, t in reg)
print(f"{'v17 regimen':<16} {h}/{len(reg)} = {h / max(len(reg), 1):.3f}")



In [ ]:

# ---------------------------------------------------------------------------
# 4. Read the output. regimen and appositive first: regimen is the freest request in
#    the set and where the joint-outcome failure lived; appositive is the highest-value
#    new construction and the most likely to come out malformed.
# ---------------------------------------------------------------------------
for frame in ["regimen", "appositive", "denial", "study_only", "mechanism_mixed",
              "effect_pd", "contradictory", "title", "coordinate", "enzyme_only",
              "out_of_scope", "sequential", "effect_protect", "advise_reason"]:
    rows = out.get(frame, [])
    if not rows:
        continue
    print(f"\n{'=' * 70}\n{frame}  ({len(rows)})\n{'=' * 70}")
    for spec, txt in rows[:5]:
        print(f"  {txt}")
        pos = [(p["between"], p["label"]) for p in spec["positives"]]
        print(f"    gold: {pos if pos else 'all NONE'}\n")

print(f"\n{'=' * 70}\nFAILURES\n{'=' * 70}")
for frame, rows in sorted(fails.items()):
    print(f"\n--- {frame} ({len(rows)}) ---")
    for why, txt in rows[:4]:
        print(f"  [{why}] {txt}")



In [ ]:

# ---------------------------------------------------------------------------
# 5. Composition and divergence. The gates are the v13-era shortcut checks; the
#    divergence table is the corpus comparison. Neither predicted F1 for v15, so read
#    them as sanity checks rather than as targets.
# ---------------------------------------------------------------------------
gates.report(inst, records=generation_records(GEN), strict=False)

from ddi import divergence as dv
print(dv.compare(train_f, inst, match_size=True).to_string(index=False))

print("""
Decide from the frame table and the read-through, not from the gates.

  every negative frame at high marker rate and near-zero avoid breaches -> generate 6000
  a frame below ~0.7 marker rate                                        -> its `says`
     rows are not landing; fix that frame rather than the whole prompt
  regimen joint-outcome rate near v14's                                 -> the empty
     says block did not fix it and the system prompt rule needs sharpening
  appositive malformed                                                  -> drop it to
     weight 0 for this run rather than losing the whole batch
""")

In [ ]:

# ===========================================================================
# Cell 3: full run, 6000 specs, ~30 min. Only if the joint-outcome rate dropped.
# ===========================================================================
GEN_FULL = "v17-full"
specs = make_specs(6000, vocab=vocab, seed=0, composition="prior")
generate_raw(specs, make_sample_fn(client, model=MODEL, reasoning_effort=EFFORT,
                                       api=API),
             gen_id=GEN_FULL, max_workers=16)

v17_id, stats = build_dataset_from_raw(
    GEN_FULL, resolver=sample_to_instances, mode="markers",
    generator={"prompt_sha": fingerprint(), "model": MODEL, "version": "v17",
               "composition": "prior",
               "note": "v17 + no joint-outcome claims in zero-assertion specs"},
    vocab_source=vocab.fingerprint(), seed=0, notes="v17, 6000 specs")
print(v17_id, stats["reject_reasons"])

v17, _ = load_dataset(v17_id)
joint_rate(v14, "v14")
joint_rate(v17, "v17")

comp = dv.compare(train_f, v14, match_size=True)[["measurement", "tier", "corpus", "synth"]]
comp = comp.rename(columns={"synth": "v14"})
comp["v17"] = dv.compare(train_f, v17, match_size=True).synth.values
print(comp.to_string(index=False))



In [ ]:

# ===========================================================================
# Cell 4: train. 3 seeds. Run after the prune sweep finishes.
# ===========================================================================
import pandas as pd
from ddi.train import train_and_eval
from ddi.experiment import log_run

BASE = {"model_name": "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext",
        "epochs": 3, "lr": 2e-5, "batch_size": 32, "max_length": 256,
        "neg_ratio": None, "render_mode": "markers"}

rows = []
for seed in [0, 1, 2]:
    cfg = {**BASE, "seed": seed, "dataset": "v16", "synth_id": v16_id}
    m = train_and_eval(cfg, v16, dev)
    log_run(cfg, m, notes="v16")
    rows.append({"seed": seed, "f1": m["micro_f1_pos"], "p": m["micro_p_pos"],
                 "r": m["micro_r_pos"]})
    print(f"v16 seed={seed} f1={m['micro_f1_pos']:.3f} p={m['micro_p_pos']:.3f} "
          f"r={m['micro_r_pos']:.3f}")

print(pd.DataFrame(rows)[["f1", "p", "r"]].agg(["mean", "std"]))
print("""
reference, same dev, 3 seeds:
  human filtered      0.790   P 0.754  R 0.829
  v14 full            0.379   P 0.302  R 0.511
  v14 judged          0.387   P 0.307  R 0.525
  v14 judged-pruned   0.408   P 0.304  R 0.623
  v15 full            0.333   P 0.272  R 0.430

v16 targets the same instances pruning removed, at source. If the mechanism is right it
should land near the pruned arm, and its recall should rise rather than its precision.
Landing near v14 unchanged means the joint-outcome phrasing was not what mattered.
""")